# Ноутбук 00 — Быстрый старт за 5 минут

Привет! Это проект **D.PRO: Клиент под защитой (Сбер)**. Задача — по данным о клиенте банка предсказать число **protection_score** (от 0 до 100): насколько хорошо клиент защищён.

В этом ноутбуке ты:
1. Поймёшь, где какие файлы лежат.
2. Проверишь, что всё установлено.
3. Одним взглядом посмотришь на данные.
4. Запустишь готовое предсказание и получишь файл для отправки.

> Остальные ноутбуки (01–06) разбирают каждый шаг подробно. Этот — только карта и быстрый результат.

Карта ноутбуков:

| Ноутбук | О чём |
| --- | --- |
| 00_quickstart | ты здесь: быстрый старт |
| 01_data_overview | смотрим на CSV-файлы глазами |
| 02_eda | графики и связи признаков с целью |
| 03_feature_engineering | новые признаки + обработка пропусков |
| 04_train_catboost | как обучается модель |
| 05_predict_submission | как делаются предсказания |
| 06_experiments | сравнение моделей: почему выбрали CatBoost |

## Шаг 0. Переходим в корень проекта

Ноутбуки лежат в папке `notebooks/`, а данные и код — на уровень выше. Первая ячейка каждого ноутбука переходит в корень, чтобы все пути (`hard_train.csv`, `artifacts/` и т.д.) работали одинаково. Запусти её (Shift+Enter).

In [ ]:
# Импортируем модуль os — он умеет узнавать текущую папку и переходить в другую.
import os
# os.getcwd() возвращает текущую папку. basename берёт её имя (последнюю часть пути).
# Если имя папки == notebooks, значит ноутбук запущен из папки notebooks — поднимаемся выше.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # chdir(..) = перейти в родительскую папку (корень проекта).
# Печатаем, где мы теперь оказались, чтобы убедиться, что всё правильно.
print("Рабочая папка:", os.getcwd())

## Шаг 1. Смотрим, какие файлы есть в проекте

In [ ]:
# Path — удобный объект для работы с путями (проверка существования файлов и т.п.).
from pathlib import Path
# Список самых важных файлов проекта: что за файл и зачем он нужен.
important_files = {
    "hard_train.csv": "обучающие данные (10 000 клиентов С ответами)",
    "hard_test.csv": "открытый тест (2 000 клиентов БЕЗ ответов)",
    "sample_submission.csv": "пример файла для отправки (правильный формат)",
    "train.py": "скрипт обучения финальной модели",
    "predict.py": "скрипт предсказания (без обучения)",
    "model_utils.py": "общие функции обработки признаков",
    "artifacts/model_metadata.json": "описание уже обученной модели",
}
# Проходим по словарю: для каждого файла проверяем, существует ли он.
for filename, description in important_files.items():
    exists = Path(filename).exists()  # True — файл есть, False — файла нет.
    mark = "OK  " if exists else "НЕТ "  # Галочка-текст: OK или НЕТ.
    print(f"{mark} {filename} — {description}")  # f-строка подставляет значения в текст.

## Шаг 2. Проверяем, что библиотеки установлены

Если какого-то импорта не хватает — выполни в терминале (не в ноутбуке):

```
python -m pip install -r requirements.txt
```

Ячейка ниже проверяет импорты и сразу показывает версии.

In [ ]:
# Пробуем импортировать все нужные библиотеки. Если какой-то нет — увидим понятную ошибку.
import numpy  # математика и массивы
import pandas  # таблицы (DataFrame) и чтение CSV
import sklearn  # классические ML-модели и утилиты
import scipy  # научная математика (оптимизация, статистики)
import catboost  # градиентный бустинг (наша финальная модель)
import lightgbm  # ещё один бустинг (для сравнения моделей)
import matplotlib  # графики
# Печатаем версии — они должны совпадать с requirements.txt (иначе цифры могут чуть отличаться).
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("scipy:", scipy.__version__)
print("catboost:", catboost.__version__)
print("Все импорты успешны!")

## Шаг 3. Смотрим на данные одним взглядом

`hard_train.csv` — это таблица: каждая строка — клиент, столбцы — признаки + ответ `protection_score`.

In [ ]:
# Читаем CSV-файл в таблицу pandas (DataFrame) — дальше работаем с переменной data.
import pandas as pd
# pd.read_csv читает файл с запятыми-разделителями. Путь — от корня проекта.
data = pd.read_csv("hard_train.csv")
# .shape — размер таблицы: (строки, столбцы). Должно быть (10000, 61).
print("Размер train:", data.shape)
# .head(3) — первые 3 строки, чтобы увидеть, как выглядят данные глазами.
data.head(3)

Колонка `customer_id` — это ID клиента (не признак!). Колонка `protection_score` — ответ, который учимся предсказывать. Всё остальное — признаки. Пустые клетки — пропуски (NaN), с ними разберёмся в ноутбуке 03.

## Шаг 4. Запускаем готовое предсказание

Модель уже обучена, её веса лежат в папке `artifacts/`. Скрипт `predict.py` загрузит веса и предскажет `hard_test.csv`. В ноутбуке shell-команды начинаются с `!`. `sys.executable` — это путь к тому же Python, который выполняет ноутбук (так команда точно найдёт установленные библиотеки).

In [ ]:
# Импортируем sys, чтобы узнать путь к текущему Python (sys.executable).
import sys
# Восклицательный знак в начале = выполнить команду в терминале, а не в Python.
# {sys.executable} в фигурных скобках подставит путь к Python автоматически.
# --input-csv hard_test.csv — какой файл предсказать; --output-csv /tmp/... — куда сохранить.
!{sys.executable} predict.py --input-csv hard_test.csv --model-dir artifacts --output-csv /tmp/quickstart_submission.csv

Готово! Проверяем файл, который получился: первые строки, размер и диапазон предсказаний.

In [ ]:
# Читаем получившийся submission-файл в таблицу.
submission = pd.read_csv("/tmp/quickstart_submission.csv")
# .shape — должно быть (2000, 2): 2000 клиентов, 2 колонки.
print("Размер submission:", submission.shape)
# .head() — первые 5 строк по умолчанию.
print(submission.head())
# Проверяем диапазон предсказаний: минимум и максимум должны быть внутри [0, 100].
print("Минимум:", submission["protection_score"].min())
print("Максимум:", submission["protection_score"].max())
# Проверяем, что нет пропусков: .isna().sum().sum() = число пустых клеток во всей таблице (должно быть 0).
print("Пропусков:", submission.isna().sum().sum())

## Что дальше?

Ты только что:
- проверил окружение,
- посмотрел на данные,
- получил готовый файл предсказаний.

Теперь иди по порядку:
1. **01_data_overview** — разберём каждый столбец данных.
2. **02_eda** — графики и поиск связей.
3. **03_feature_engineering** — новые признаки и пропуски (самая важная механика!).
4. **04_train_catboost** — обучение модели.
5. **05_predict_submission** — предсказание изнутри.
6. **06_experiments** — почему CatBoost, а не что-то другое.

Полное обучение запускается командой (несколько минут, делает новые веса): `python train.py --train-csv hard_train.csv --model-dir artifacts --seed 434089`. Но сначала — ноутбуки, чтобы понимать, что там происходит!